In [1]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

print(f"PyTorch versão {torch.__version__} importado com sucesso!")

# Definindo o dispositivo (Usa GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cpu'
print(f"Dispositivo de processamento: {device}")

PyTorch versão 2.8.0 importado com sucesso!
Dispositivo de processamento: cpu


In [2]:
df = pd.read_csv('HateBR.csv', sep=',', usecols=['comentario', 'label_final'])

X = df['comentario'].values
y = df['label_final'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f'Tamanho do treino: {len(X_train)} amostras')
print(f'Tamanho do teste: {len(X_test)}')

Tamanho do treino: 5250 amostras
Tamanho do teste: 1750


In [3]:
MAX_LEN = 15 # Tamanho máximo da frase

# 1. Construindo o vocabulário a partir dos dados de treino
todas_as_palavras = [palavra for frase in X_train for palavra in frase.lower().split()]
frequencia_palavras = Counter(todas_as_palavras)

# Mapeando palavras para índicies. Reservamos 0 para Padding e 1 para Unknown (OOV)
vocabulario = {palavra: idx + 2 for idx, (palavra, _) in enumerate(frequencia_palavras.most_common())}
vocabulario['<PAD>'] = 0
vocabulario['<UNK>'] = 1

# Removendo vírgulas
def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto

# Função para converter texto em sequência de números com padding
def texto_para_sequencia(texto, vocab, max_len):
    tokens = limpar_texto(texto).split()
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]

    # Padding (preenchimento com zeros) ou truncamento (corta)
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# 2. Criando a classe Dataset do PyTorch
class TextoDataset(Dataset):
    def __init__(self, X, y, vocab, max_len):
        self.X = [texto_para_sequencia(texto, vocab, max_len) for texto in X]
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Retorna a sequência como tensor de inteiros e a label como tensor float
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.float32)
# Instanciando Datasets e DataLoaders (para gerar batches)
train_dataset = TextoDataset(X_train, y_train, vocabulario, MAX_LEN)
test_dataset = TextoDataset(X_test, y_test, vocabulario, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, pin_memory=True)

print("Tamanho do vocabulário:", len(vocabulario))

Tamanho do vocabulário: 15734


In [4]:
class ModeloHateSpeechLSTM(nn.Module):
    def __init__(self, tamanho_vocab, dimensao_embedding, dimensao_oculta):
        super(ModeloHateSpeechLSTM, self).__init__()

        # Camada de Embedding: transforma inteiros em vetores densos.
        # padding_idx=0 diz ao pytorch para ignorar os zeros usados no padding
        self.embedding = nn.Embedding(tamanho_vocab, dimensao_embedding, padding_idx=0)

        # Camada LSTM: batch_first=True indica que o formato do tensor é (batch, sequencia, caracteristicas)
        self.lstm = nn.LSTM(dimensao_embedding, dimensao_oculta, batch_first=True)

        # Droput para regularização
        self.dropout = nn.Dropout(0.2)

        # Camada Totalmente Conectada (Dense) para classificação
        self.fc = nn.Linear(dimensao_oculta, 1)

        # Ativação sigmóid para transformar a saída final em probabilidade (0 a 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)

        lstm_out, (hidden, cell) = self.lstm(x)

        # A LSTM retorna todos os estados (lstm_out) e os estados finais (hidden, cell).
        # Como queremos classificar a frease toda, pegamos apenas o último hidden state
        ultimo_hidden_state = hidden[-1]

        out = self.dropout(ultimo_hidden_state)
        out = self.fc(out)
        out = self.sigmoid(out)

        return out.squeeze() # Retorna dimensões extras (ex: de [batch_size, 1] para [batch_size])

# Instanciando o modelo e movendo para a CPU/GPU
tamanho_vocab = len(vocabulario)
dim_embedding = 16
dim_oculta = 32

modelo = ModeloHateSpeechLSTM(tamanho_vocab, dim_embedding, dim_oculta).to(device)
print(modelo)

ModeloHateSpeechLSTM(
  (embedding): Embedding(15734, 16, padding_idx=0)
  (lstm): LSTM(16, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [5]:
# Função de Custo (Binary Cross Entropy) e Otimizador (Adam)
criterio = nn.BCELoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.01)

def calcular_acuracia(previsoes, labels):
    # Arredonda as previsões para 0 ou 1
    previsoes_arredondadas = torch.round(previsoes)

    # Compara quantas previsões são iguais aos rótulos reais
    corretas = (previsoes_arredondadas == labels).float()

    # Calcula a média (ex: 8 acertos em 10 = 0.8 ou 80%)
    acuracia = corretas.sum() / len(corretas)
    return acuracia

epocas = 100

for epoca in range(epocas):
    modelo.train() # coloca o modelo em modo de treinamento (ativa dropout, etc)
    loss_treino = 0.0
    acc_treino = 0.0

    for textos, labels in train_loader:
        textos, labels = textos.to(device), labels.to(device)

        otimizador.zero_grad()
        previsoes = modelo(textos)

        loss = criterio(previsoes, labels)
        acc = calcular_acuracia(previsoes, labels)
        loss.backward()
        otimizador.step()

        loss_treino += loss.item()
        acc_treino += acc.item()

    loss_media = loss_treino / len(train_loader)
    acc_media = acc_treino / len(train_loader)

    if(epoca+1) % 2 == 0 or epoca == 0:
        print(f"Época {epoca+1}/{epocas} | Loss: {loss_media:.4f} | Acurácia: {acc_media*100:.2f}%")

/Users/amandafrolini/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Época 1/100 | Loss: 0.6905 | Acurácia: 52.90%
Época 2/100 | Loss: 0.6393 | Acurácia: 63.54%
Época 4/100 | Loss: 0.3779 | Acurácia: 84.52%
Época 6/100 | Loss: 0.1890 | Acurácia: 93.59%
Época 8/100 | Loss: 0.0904 | Acurácia: 97.10%
Época 10/100 | Loss: 0.0596 | Acurácia: 98.29%
Época 12/100 | Loss: 0.0307 | Acurácia: 99.16%
Época 14/100 | Loss: 0.0149 | Acurácia: 99.67%
Época 16/100 | Loss: 0.0157 | Acurácia: 99.61%
Época 18/100 | Loss: 0.0186 | Acurácia: 99.46%
Época 20/100 | Loss: 0.0169 | Acurácia: 99.57%
Época 22/100 | Loss: 0.0086 | Acurácia: 99.81%
Época 24/100 | Loss: 0.0060 | Acurácia: 99.83%
Época 26/100 | Loss: 0.0153 | Acurácia: 99.57%
Época 28/100 | Loss: 0.0054 | Acurácia: 99.83%
Época 30/100 | Loss: 0.0117 | Acurácia: 99.72%
Época 32/100 | Loss: 0.0046 | Acurácia: 99.89%
Época 34/100 | Loss: 0.0022 | Acurácia: 99.94%
Época 36/100 | Loss: 0.0025 | Acurácia: 99.89%
Época 38/100 | Loss: 0.0011 | Acurácia: 99.96%
Época 40/100 | Loss: 0.0011 | Acurácia: 99.96%
Época 42/100 | Los

In [6]:
# Coloca o modelo em modo de avaliação (desliga Dropout, etc.)
modelo.eval()

loss_teste = 0.0
acc_teste = 0.0

# torch.no_grad() diz ao PyTorch para não calcular gradientes aqui,
# economizando muita memória e processamento.
with torch.no_grad():
    for textos, labels in test_loader:
        textos, labels = textos.to(device), labels.to(device)

        previsoes = modelo(textos)
        loss = criterio(previsoes, labels)
        acc = calcular_acuracia(previsoes, labels)

        loss_teste += loss.item()
        acc_teste += acc.item()

loss_teste_media = loss_teste / len(test_loader)
acc_teste_media = acc_teste / len(test_loader)

print("-" * 30)
print("RESULTADOS NO CONJUNTO DE TESTE:")
print(f"Loss: {loss_teste_media:.4f}")
print(f"Acurácia: {acc_teste_media*100:.2f}%")
print("-" * 30)

------------------------------
RESULTADOS NO CONJUNTO DE TESTE:
Loss: 2.0778
Acurácia: 79.45%
------------------------------


In [7]:
def detectar_odio(frase, modelo, vocab):
    modelo.eval() # Coloca o modelo em modo de avaliação (desliga dropout)

    # Processamento manual ed frase
    seq = texto_para_sequencia(frase, vocab, MAX_LEN)

    # Converte para tensor, adiciona dimensão de batch [1, MAX_LEN] e joga para o device
    tensor_entrada = torch.tensor([seq], dtype=torch.long).to(device)

    # Evita calcular gradientes durante a interferência (economiza memória e tempo)
    with torch.no_grad():
        predicao = modelo(tensor_entrada).item()

    if predicao > 0.5:
        resultado = "DISCURSO DE ÓDIO DETECTADO!!!"
    else:
        resultado = "TEXTO NORMAL/NEUTRO"

    print(f"Frase: '{frase}'")
    print(f"Predição Bruta: {predicao:.4f}")
    print(f"Classificação: {resultado}\n")

    # Testes práticos
detectar_odio("eu acho que vocês deveriam todos morrer", modelo, vocabulario)
detectar_odio("que dia excelente para estudar programação", modelo, vocabulario)
detectar_odio("eu odeio quando acaba a luz", modelo, vocabulario)
detectar_odio("meu dia fica péssimo quando esqueço o guarda chuva", modelo, vocabulario)
detectar_odio("eu odeio pessoas assim, do seu partido", modelo, vocabulario)
detectar_odio("eu sou comunista", modelo, vocabulario)
detectar_odio("eu sou capitalista", modelo, vocabulario)

Frase: 'eu acho que vocês deveriam todos morrer'
Predição Bruta: 0.0000
Classificação: TEXTO NORMAL/NEUTRO

Frase: 'que dia excelente para estudar programação'
Predição Bruta: 0.0000
Classificação: TEXTO NORMAL/NEUTRO

Frase: 'eu odeio quando acaba a luz'
Predição Bruta: 1.0000
Classificação: DISCURSO DE ÓDIO DETECTADO!!!

Frase: 'meu dia fica péssimo quando esqueço o guarda chuva'
Predição Bruta: 0.1417
Classificação: TEXTO NORMAL/NEUTRO

Frase: 'eu odeio pessoas assim, do seu partido'
Predição Bruta: 1.0000
Classificação: DISCURSO DE ÓDIO DETECTADO!!!

Frase: 'eu sou comunista'
Predição Bruta: 1.0000
Classificação: DISCURSO DE ÓDIO DETECTADO!!!

Frase: 'eu sou capitalista'
Predição Bruta: 0.0000
Classificação: TEXTO NORMAL/NEUTRO

